<a href="https://colab.research.google.com/github/abilashkannanv/AIML/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
import os
import openai
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = "voc-262423617156170395256969020d2c765580.35946547"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [55]:
!pip install langchain-openai
from langchain_openai import ChatOpenAI

In [56]:
!pip install langchain-community
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

In [57]:
import os
!pip install pypdf
from pypdf import PdfReader

pdf_names = ['AIML.pdf', 'AIMLDL.pdf', 'ML1.pdf', 'ML.pdf', 'Chunking.pdf', 'RAG.pdf']
base_directory = '/content/'

raw_documents = []
for pdf_file in pdf_names:
    file_path = os.path.join(base_directory, pdf_file)
    try:
        loader = PyPDFLoader(file_path)
        docs = loader.load()
        for doc in docs:
            doc.metadata['doc_title'] = pdf_file.replace('.pdf', '')
        raw_documents.extend(docs)
        print(f"Successfully loaded {pdf_file}")
    except Exception as e:
        print(f"Error loading {pdf_file}: {e}")

Successfully loaded AIML.pdf
Successfully loaded AIMLDL.pdf
Successfully loaded ML1.pdf
Successfully loaded ML.pdf
Successfully loaded Chunking.pdf
Successfully loaded RAG.pdf


Now, let's chunk the documents using `RecursiveCharacterTextSplitter` with the specified `chunk_size` and `chunk_overlap`.

In [58]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 600
chunk_overlap = int(chunk_size * 0.15) # 15% overlap

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(raw_documents)

# Add 1-indexed page number and chunk_id to metadata
for i, chunk in enumerate(chunks):
    if 'page' in chunk.metadata:
        chunk.metadata['page'] = chunk.metadata['page'] + 1 # Convert to 1-indexed page
    else:
        chunk.metadata['page'] = 'N/A' # Handle cases where page metadata might be missing
    chunk.metadata['chunk_id'] = i

print(f"Created {len(chunks)} chunks.")
print("First chunk metadata:")
display(chunks[0].metadata)
print("First chunk content preview:")
display(chunks[0].page_content[:200] + "...")

Created 4455 chunks.
First chunk metadata:


{'producer': '3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)',
 'creator': 'Microsoft® Word 2016',
 'creationdate': '2022-08-12T08:58:55+00:00',
 'author': 'cvrpexam',
 'moddate': '2024-06-12T14:26:27+00:00',
 'source': '/content/AIML.pdf',
 'total_pages': 93,
 'page': 1,
 'page_label': '1',
 'doc_title': 'AIML',
 'start_index': 0,
 'chunk_id': 0}

First chunk content preview:


'1 \nSTUDY MATERIAL \n \n \nOn  \n \n \nArtificial intelligence \nand Machine Learning \n \n(For 6th Semester CSE,CVRP) \n \n \n \n \n \n \n \n \nPrepared by : \n \n PRANGYA PARAMITA MOHAPATRA,ASST.PROF.(CSE),CVRP...'

### 1. Initialize Embeddings and Vector Store

First, we'll initialize our embedding model. We'll use `OpenAIEmbeddings` for this purpose. Then, we'll create a FAISS vector store from our document `chunks` using these embeddings. FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors.

In [59]:
!pip install faiss-cpu

from langchain_openai import OpenAIEmbeddings
import faiss
from langchain_community.vectorstores import FAISS

# Initialize the embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key = "voc-2033465586156170068dbe3820ba1c1.93790491",
                    openai_api_base = "https://openai.vocareum.com/v1")

# Create a FAISS vector store from the chunks
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"FAISS vector store created with {vectorstore.index.ntotal} documents.")

FAISS vector store created with 4455 documents.


### 2. Initialize Retriever

Next, we'll create a retriever from our FAISS vector store. The retriever's job is to fetch relevant documents based on a given query.

In [60]:
retriever = vectorstore.as_retriever()
print("Retriever initialized.")

Retriever initialized.


### 3. Define Prompt Template

Now, we'll define a prompt template that will guide the LLM to provide accurate and cited answers based on the retrieved context.

In [61]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks. \n
Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. \n
Use three sentences maximum and keep the answer concise.\n
Question: {question} \n
Context: {context} \n
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

print("Prompt template defined.")

Prompt template defined.


### 4.1. Prompt Template Variant P1 (Concise)

This prompt template (`prompt_p1`) is designed for concise answers (<= 120 tokens), strictly using the provided context, and including citations in the format `[doc_title#page]`.

In [62]:
from langchain_core.prompts import ChatPromptTemplate

template_p1 = """You are an assistant for question-answering tasks. \n
Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. \n
Answer in <= 120 tokens. Provide citations in the format [doc_title#page] for each piece of information.

Question: {question} \n
Context: {context} \n
Answer:"""

prompt_p1 = ChatPromptTemplate.from_template(template_p1)

print("Prompt template P1 (concise) defined.")

Prompt template P1 (concise) defined.


### 4.2. Prompt Template Variant P2 (Reasoned)

This prompt template (`prompt_p2`) extends P1 by asking the LLM to first list relevant evidence snippets and then provide its concise answer, maintaining the token limit and citation format.

In [63]:
from langchain_core.prompts import ChatPromptTemplate

template_p2 = """You are an assistant for question-answering tasks. \n
Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. \n
Answer in <= 120 tokens. Provide citations in the format [doc_title#page] for each piece of information.
First, briefly list the 2 most relevant supporting snippets from the context under an "Evidence:" bullet point. Then, provide your concise answer under an "Answer:" bullet point.

Question: {question} \n
Context: {context} \n
Answer:"""

prompt_p2 = ChatPromptTemplate.from_template(template_p2)

print("Prompt template P2 (reasoned) defined.")

Prompt template P2 (reasoned) defined.


### 4. Construct RAG Chain

Finally, we'll assemble all the components: the retriever, the LLM, and the prompt template, into a RAG chain using LangChain's expression language. This chain will take a question, retrieve relevant documents, format them into the prompt, and then pass the prompt to the LLM to generate an answer.

In [64]:
import os
import openai
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = "voc-2033465586156170068dbe3820ba1c1.93790491"

from langchain_openai import ChatOpenAI

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

my_llm = ChatOpenAI(model_name = "gpt-4o-mini", api_key = "voc-2033465586156170068dbe3820ba1c1.93790491",
                    base_url = "https://openai.vocareum.com/v1")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | my_llm
    | StrOutputParser()
)

print("RAG chain constructed.")

RAG chain constructed.


### 5. Generate Response

Now you can ask a question and get a generated answer from the RAG chain.

In [65]:
question = "What are some key aspects of Chunking?"
response = rag_chain.invoke(question)
print(response)

Chunking is the process of breaking down large text into smaller, manageable segments called chunks to optimize relevance for storage in a vector database. It requires balancing chunk size to ensure they are meaningful yet small enough for efficient processing in applications like retrieval augmented generation. There are different methods for chunking, including fixed-size chunking based on the maximum context window of an embedding model.


### 1. Define Evaluation Questions

To systematically evaluate the performance of our RAG chain variants, we'll use a set of diverse evaluation questions.

In [66]:
evaluation_questions = [
    "What is the definition of Artificial Intelligence?",
    "How does a Recursive Character Text Splitter work?",
    "What are the main benefits of using a Retrieval Augmented Generation (RAG) system?",
    "Explain the concept of cosine similarity in vector spaces.",
    "What is the purpose of chunking in RAG?",
    "When was the Turing Test introduced and by whom?",
    "What is the capital of France?", # Boundary question
    "Who won the 2024 NBA championship?", # Boundary question
    "Based on the explanation of AI, what are some ethical concerns associated with its development?", # Multi-hop/Follow-up (will be treated as standalone)
    "Considering the benefits of RAG, how does it compare to a traditional chatbot without retrieval capabilities?" # Multi-hop/Follow-up (will be treated as standalone)
]

print("Evaluation questions defined:")
for i, q in enumerate(evaluation_questions):
    print(f"{i+1}. {q}")

Evaluation questions defined:
1. What is the definition of Artificial Intelligence?
2. How does a Recursive Character Text Splitter work?
3. What are the main benefits of using a Retrieval Augmented Generation (RAG) system?
4. Explain the concept of cosine similarity in vector spaces.
5. What is the purpose of chunking in RAG?
6. When was the Turing Test introduced and by whom?
7. What is the capital of France?
8. Who won the 2024 NBA championship?
9. Based on the explanation of AI, what are some ethical concerns associated with its development?
10. Considering the benefits of RAG, how does it compare to a traditional chatbot without retrieval capabilities?


### 2. Create Experiments Grid

Below is the experiment grid, detailing the different configurations we will test for chunking parameters, embedding models, and prompt templates. Each run varies one factor from a base configuration (chunk_size=600, overlap=15%, OpenAI embeddings, P1 prompt) to allow for A/B style comparison.

In [67]:
experiments_grid = [
    {
        "run_id": "Base-P1-OpenAI-C600-O15",
        "chunk_size": 600,
        "overlap_percentage": 0.15,
        "embeddings_type": "OpenAI",
        "prompt_template_name": "P1",
        "description": "Base configuration with OpenAI embeddings and P1 prompt."
    },
    {
        "run_id": "CS-300-P1-OpenAI-O15",
        "chunk_size": 300,
        "overlap_percentage": 0.15,
        "embeddings_type": "OpenAI",
        "prompt_template_name": "P1",
        "description": "Vary chunk size to 300, keeping other base configs."
    },
    {
        "run_id": "CS-1000-P1-OpenAI-O15",
        "chunk_size": 1000,
        "overlap_percentage": 0.15,
        "embeddings_type": "OpenAI",
        "prompt_template_name": "P1",
        "description": "Vary chunk size to 1000, keeping other base configs."
    },
    {
        "run_id": "Overlap-10-P1-OpenAI-C600",
        "chunk_size": 600,
        "overlap_percentage": 0.10,
        "embeddings_type": "OpenAI",
        "prompt_template_name": "P1",
        "description": "Vary overlap to 10%, keeping other base configs."
    },
    {
        "run_id": "Overlap-20-P1-OpenAI-C600",
        "chunk_size": 600,
        "overlap_percentage": 0.20,
        "embeddings_type": "OpenAI",
        "prompt_template_name": "P1",
        "description": "Vary overlap to 20%, keeping other base configs."
    },
    {
        "run_id": "P2-OpenAI-C600-O15",
        "chunk_size": 600,
        "overlap_percentage": 0.15,
        "embeddings_type": "OpenAI",
        "prompt_template_name": "P2",
        "description": "Vary prompt to P2, keeping other base configs."
    }
]

import pandas as pd

display(pd.DataFrame(experiments_grid))

,run_id,chunk_size,overlap_percentage,embeddings_type,prompt_template_name,description
0,Base-P1-OpenAI-C600-O15,600,0.15,OpenAI,P1,Base configuration with OpenAI embeddings and ...
1,CS-300-P1-OpenAI-O15,300,0.15,OpenAI,P1,"Vary chunk size to 300, keeping other base con..."
2,CS-1000-P1-OpenAI-O15,1000,0.15,OpenAI,P1,"Vary chunk size to 1000, keeping other base co..."
3,Overlap-10-P1-OpenAI-C600,600,0.10,OpenAI,P1,"Vary overlap to 10%, keeping other base configs."
4,Overlap-20-P1-OpenAI-C600,600,0.20,OpenAI,P1,"Vary overlap to 20%, keeping other base configs."
5,P2-OpenAI-C600-O15,600,0.15,OpenAI,P2,"Vary prompt to P2, keeping other base configs."


### 3. Construct RAG Chain with P1 (Concise)

In [68]:
rag_chain_p1 = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_p1
    | my_llm
    | StrOutputParser()
)

print("RAG chain with P1 prompt constructed.")

RAG chain with P1 prompt constructed.


### 4. Construct RAG Chain with P2 (Reasoned)

In [69]:
rag_chain_p2 = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_p2
    | my_llm
    | StrOutputParser()
)

print("RAG chain with P2 prompt constructed.")

RAG chain with P2 prompt constructed.


### 5. Comprehensive Experiment Runner and Metric Collection

This section implements a detailed experiment loop that iterates through the `experiments_grid`. For each configuration, it dynamically sets up the text splitter, creates document chunks, builds a FAISS vector store, and configures the appropriate RAG chain. It then runs all `evaluation_questions`, collecting comprehensive metrics for each answer, including `run_id`, `question`, `answer`, `citations` (from retrieved documents and extracted from LLM output), `used_chunks`, approximate `token_count`, and `latency`. Finally, all results are stored in a Pandas DataFrame and saved to a CSV file.

In [71]:
import time
import re
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.callbacks import get_openai_callback

# Ensure the API key is accessible for dynamic LLM/Embeddings creation
# It's already loaded globally as openai.api_key

def extract_citations_from_response(text):
    """Extracts citations in [doc_title#page] format from a given text."""
    # Regex to find patterns like [doc_title#page] or [doc_title#page.subpage]
    # It's flexible to handle various page formats
    citations = re.findall(r'\[([A-Za-z0-9_ -]+?)#([0-9A-Za-z.]+?)\]', text)
    formatted_citations = [f'{title}#{page}' for title, page in citations]
    return "; ".join(sorted(list(set(formatted_citations))))

all_experiment_results = []

for experiment_config in experiments_grid:
    run_id = experiment_config['run_id']
    current_chunk_size = experiment_config['chunk_size']
    current_overlap_percentage = experiment_config['overlap_percentage']
    current_embeddings_type = experiment_config['embeddings_type']
    current_prompt_template_name = experiment_config['prompt_template_name']

    print(f"\n--- Running Experiment: {run_id} ---")
    print(f"  Chunk Size: {current_chunk_size}, Overlap: {current_overlap_percentage*100}%")
    print(f"  Embeddings: {current_embeddings_type}, Prompt: {current_prompt_template_name}")

    # 1. Dynamically configure Text Splitter
    current_chunk_overlap = int(current_chunk_size * current_overlap_percentage)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=current_chunk_size,
        chunk_overlap=current_chunk_overlap,
        length_function=len,
        add_start_index=True,
    )
    current_chunks = text_splitter.split_documents(raw_documents)
    for i, chunk in enumerate(current_chunks):
        if 'page' in chunk.metadata:
            chunk.metadata['page'] = chunk.metadata['page'] + 1
        else:
            chunk.metadata['page'] = 'N/A'
        chunk.metadata['chunk_id'] = i
    print(f"  Created {len(current_chunks)} chunks for this configuration.")

    # 2. Dynamically configure Embeddings and Vector Store
    if current_embeddings_type == "OpenAI":
        current_embeddings = OpenAIEmbeddings(
            model="text-embedding-3-small",
            openai_api_key=openai.api_key,
            openai_api_base="https://openai.vocareum.com/v1"
        )
    else:
        print(f"Warning: Unknown embeddings_type '{current_embeddings_type}'. Using OpenAI as fallback.")
        current_embeddings = OpenAIEmbeddings(
            model="text-embedding-3-small",
            openai_api_key=openai.api_key,
            openai_api_base="https://openai.vocareum.com/v1"
        )

    current_vectorstore = FAISS.from_documents(current_chunks, current_embeddings)
    current_retriever = current_vectorstore.as_retriever()
    print(f"  FAISS vector store created for {current_embeddings_type} embeddings.")

    # 3. Select Prompt Template
    if current_prompt_template_name == "P1":
        current_prompt = prompt_p1
    elif current_prompt_template_name == "P2":
        current_prompt = prompt_p2
    else:
        current_prompt = prompt # Fallback to default if name is not P1 or P2
    print(f"  Using prompt template: {current_prompt_template_name}")

    # 4. Define the RAG chain for the current configuration
    # The format_docs function is used for context, but we need to also get the raw docs from the retriever
    current_rag_chain = (
        {"context": current_retriever | format_docs, "question": RunnablePassthrough()}
        | current_prompt
        | my_llm
        | StrOutputParser()
    )

    # 5. Run evaluation questions and collect detailed metrics
    for i, question in enumerate(evaluation_questions):
        print(f"    Query {i+1}: {question}")
        start_time = time.time()

        retrieved_docs_for_question = current_retriever.invoke(question)
        formatted_context_for_llm = "\n\n".join(doc.page_content for doc in retrieved_docs_for_question)

        # Extract citations and used_chunks from retrieved documents
        citations_from_docs = []
        used_chunks_details = [] # Store unique identifiers for chunks
        for doc in retrieved_docs_for_question:
            title = doc.metadata.get('doc_title', 'UnknownDoc')
            page = doc.metadata.get('page', 'N/A')
            chunk_id = doc.metadata.get('chunk_id', 'N/A')
            citations_from_docs.append(f"{title}#{page}")
            used_chunks_details.append(f"{title}#{chunk_id}")

        # Get unique and sorted citations/chunk details
        citations_from_docs_str = "; ".join(sorted(list(set(citations_from_docs))))
        used_chunks_str = "; ".join(sorted(list(set(used_chunks_details))))

        # Invoke LLM and measure tokens/cost using callback
        answer = ""
        total_tokens = 0
        with get_openai_callback() as cb:
            answer = current_rag_chain.invoke(question)
            total_tokens = cb.total_tokens # Includes prompt + completion tokens
            # Note: LangChain's get_openai_callback total_cost is not directly available for the configured base_url.
            # For vocareum, it might require specific setup or manual calculation based on token usage.
            # We will capture tokens, but cost might not be accurate or available directly.

        end_time = time.time()
        latency = end_time - start_time

        llm_generated_citations = extract_citations_from_response(answer)

        all_experiment_results.append({
            "run_id": run_id,
            "question": question,
            "answer": answer,
            "citations_from_retrieved_docs": citations_from_docs_str,
            "llm_extracted_citations": llm_generated_citations,
            "used_chunks": used_chunks_str,
            "token_count": total_tokens,
            "latency_seconds": latency
        })
        # print(f"      Response: {answer[:100]}...") # Preview of the answer
        # print(f"      Latency: {latency:.2f}s, Tokens: {total_tokens}")

print("\nAll experiments completed and results collected.")


--- Running Experiment: Base-P1-OpenAI-C600-O15 ---
  Chunk Size: 600, Overlap: 15.0%
  Embeddings: OpenAI, Prompt: P1
  Created 4455 chunks for this configuration.
  FAISS vector store created for OpenAI embeddings.
  Using prompt template: P1
    Query 1: What is the definition of Artificial Intelligence?
    Query 2: How does a Recursive Character Text Splitter work?
    Query 3: What are the main benefits of using a Retrieval Augmented Generation (RAG) system?
    Query 4: Explain the concept of cosine similarity in vector spaces.
    Query 5: What is the purpose of chunking in RAG?
    Query 6: When was the Turing Test introduced and by whom?
    Query 7: What is the capital of France?
    Query 8: Who won the 2024 NBA championship?
    Query 9: Based on the explanation of AI, what are some ethical concerns associated with its development?
    Query 10: Considering the benefits of RAG, how does it compare to a traditional chatbot without retrieval capabilities?

--- Running Exper

In [72]:
import pandas as pd

# Create DataFrame from collected results
results_df_comprehensive = pd.DataFrame(all_experiment_results)

# Display the first few rows of the comprehensive results
print("Comprehensive Experiment Results (first 5 rows):")
display(results_df_comprehensive.head())

# Save results to CSV
output_csv_path = "rag_experiment_results.csv"
results_df_comprehensive.to_csv(output_csv_path, index=False)
print(f"\nFull experiment results saved to {output_csv_path}")

Comprehensive Experiment Results (first 5 rows):


,run_id,question,answer,citations_from_retrieved_docs,llm_extracted_citations,used_chunks,token_count,latency_seconds
0,Base-P1-OpenAI-C600-O15,What is the definition of Artificial Intellige...,Artificial Intelligence (AI) is defined as the...,AIML#10; AIML#2; AIMLDL#21; AIMLDL#23,doc_title#page,AIML#1; AIML#33; AIMLDL#338; AIMLDL#347,592,4.177359
1,Base-P1-OpenAI-C600-O15,How does a Recursive Character Text Splitter w...,A Recursive Character Text Splitter works by s...,Chunking#4; ML#85,doc_title#page,Chunking#4282; Chunking#4284; Chunking#4285; M...,652,13.235872
2,Base-P1-OpenAI-C600-O15,What are the main benefits of using a Retrieva...,The main benefits of using a Retrieval Augment...,RAG#2; RAG#4,doc_title#page,RAG#4304; RAG#4307; RAG#4308; RAG#4315,632,3.873564
3,Base-P1-OpenAI-C600-O15,Explain the concept of cosine similarity in ve...,Cosine similarity is a measure of similarity b...,AIMLDL#180; ML#10; ML#55; ML#82,doc_title#page,AIMLDL#906; ML#3792; ML#3993; ML#4101,678,5.046947
4,Base-P1-OpenAI-C600-O15,What is the purpose of chunking in RAG?,The purpose of chunking in Retrieval-Augmented...,Chunking#1; Chunking#3; Chunking#4; Chunking#5,context#1,Chunking#4264; Chunking#4280; Chunking#4286; C...,602,6.501761



Full experiment results saved to rag_experiment_results.csv


In [75]:
import pandas as pd

# Load the experiment results
output_csv_path = "rag_experiment_results.csv"
try:
    evaluation_df = pd.read_csv(output_csv_path)
    print(f"Successfully loaded {len(evaluation_df)} results from {output_csv_path}")
except FileNotFoundError:
    print(f"Error: The file {output_csv_path} was not found. Please ensure the comprehensive experiment runner (cell `f531e361`) has been successfully executed.")
    evaluation_df = pd.DataFrame() # Create an empty DataFrame to avoid further errors

# Display the columns to verify data
if not evaluation_df.empty:
    print("\nColumns in loaded DataFrame:")
    print(evaluation_df.columns.tolist())
    display(evaluation_df.head())

Successfully loaded 60 results from rag_experiment_results.csv

Columns in loaded DataFrame:
['run_id', 'question', 'answer', 'citations_from_retrieved_docs', 'llm_extracted_citations', 'used_chunks', 'token_count', 'latency_seconds']


,run_id,question,answer,citations_from_retrieved_docs,llm_extracted_citations,used_chunks,token_count,latency_seconds
0,Base-P1-OpenAI-C600-O15,What is the definition of Artificial Intellige...,Artificial Intelligence (AI) is defined as the...,AIML#10; AIML#2; AIMLDL#21; AIMLDL#23,doc_title#page,AIML#1; AIML#33; AIMLDL#338; AIMLDL#347,592,4.177359
1,Base-P1-OpenAI-C600-O15,How does a Recursive Character Text Splitter w...,A Recursive Character Text Splitter works by s...,Chunking#4; ML#85,doc_title#page,Chunking#4282; Chunking#4284; Chunking#4285; M...,652,13.235872
2,Base-P1-OpenAI-C600-O15,What are the main benefits of using a Retrieva...,The main benefits of using a Retrieval Augment...,RAG#2; RAG#4,doc_title#page,RAG#4304; RAG#4307; RAG#4308; RAG#4315,632,3.873564
3,Base-P1-OpenAI-C600-O15,Explain the concept of cosine similarity in ve...,Cosine similarity is a measure of similarity b...,AIMLDL#180; ML#10; ML#55; ML#82,doc_title#page,AIMLDL#906; ML#3792; ML#3993; ML#4101,678,5.046947
4,Base-P1-OpenAI-C600-O15,What is the purpose of chunking in RAG?,The purpose of chunking in Retrieval-Augmented...,Chunking#1; Chunking#3; Chunking#4; Chunking#5,context#1,Chunking#4264; Chunking#4280; Chunking#4286; C...,602,6.501761


In [74]:
if not evaluation_df.empty:
    # Initialize columns for manual scoring
    evaluation_df['Relevance_Score'] = ''
    evaluation_df['Groundedness_Score'] = ''
    evaluation_df['Citation_Correctness_Score'] = ''

    # Automated Conciseness Score
    evaluation_df['Conciseness_Score'] = evaluation_df['token_count'].apply(lambda x:
        2 if x <= 120 else (1 if x <= 150 else 0)
    )

    # Automated Refusal Correctness for Boundary Questions
    boundary_questions = [
        "What is the capital of France?",
        "Who won the 2024 NBA championship?"
    ]

    def check_refusal(row):
        if row['question'] in boundary_questions:
            answer_lower = row['answer'].lower()
            if "i don't know" in answer_lower or "not know" in answer_lower or "cannot answer" in answer_lower or "not found" in answer_lower or "no information" in answer_lower:
                return 'Yes'
            else:
                return 'No'
        return 'N/A'

    evaluation_df['Refusal_Correct'] = evaluation_df.apply(check_refusal, axis=1)

    # Memory Continuity - mark as N/A due to current RAG chain design
    evaluation_df['Memory_Continuity_Score'] = 'N/A'

    print("\nDataFrame prepared for evaluation:")
    # Display relevant columns for easy manual scoring
    display(evaluation_df[['run_id', 'question', 'answer', 'citations_from_retrieved_docs', 'llm_extracted_citations', 'token_count', 'Relevance_Score', 'Groundedness_Score', 'Citation_Correctness_Score', 'Conciseness_Score', 'Refusal_Correct', 'Memory_Continuity_Score']].head(10))

    # Optionally save this prepared DataFrame for external manual review
    evaluation_df.to_csv('rag_evaluation_scores_template.csv', index=False)
    print("\nA template for manual evaluation has been saved to `rag_evaluation_scores_template.csv`.")
else:
    print("Cannot prepare evaluation DataFrame because `evaluation_df` is empty.")


DataFrame prepared for evaluation:


,run_id,question,answer,citations_from_retrieved_docs,llm_extracted_citations,token_count,Relevance_Score,Groundedness_Score,Citation_Correctness_Score,Conciseness_Score,Refusal_Correct,Memory_Continuity_Score
0,Base-P1-OpenAI-C600-O15,What is the definition of Artificial Intellige...,Artificial Intelligence (AI) is defined as the...,AIML#10; AIML#2; AIMLDL#21; AIMLDL#23,doc_title#page,592,,,,0,N/A,N/A
1,Base-P1-OpenAI-C600-O15,How does a Recursive Character Text Splitter w...,A Recursive Character Text Splitter works by s...,Chunking#4; ML#85,doc_title#page,652,,,,0,N/A,N/A
2,Base-P1-OpenAI-C600-O15,What are the main benefits of using a Retrieva...,The main benefits of using a Retrieval Augment...,RAG#2; RAG#4,doc_title#page,632,,,,0,N/A,N/A
3,Base-P1-OpenAI-C600-O15,Explain the concept of cosine similarity in ve...,Cosine similarity is a measure of similarity b...,AIMLDL#180; ML#10; ML#55; ML#82,doc_title#page,678,,,,0,N/A,N/A
4,Base-P1-OpenAI-C600-O15,What is the purpose of chunking in RAG?,The purpose of chunking in Retrieval-Augmented...,Chunking#1; Chunking#3; Chunking#4; Chunking#5,context#1,602,,,,0,N/A,N/A
5,Base-P1-OpenAI-C600-O15,When was the Turing Test introduced and by whom?,The Turing Test was introduced by Alan Turing ...,AIML#15; AIML#2; AIMLDL#24; AIMLDL#25,doc_title#15,663,,,,0,N/A,N/A
6,Base-P1-OpenAI-C600-O15,What is the capital of France?,The capital of France is Paris.,AIMLDL#244; AIMLDL#300; AIMLDL#302; AIMLDL#318,NaN,597,,,,0,No,N/A
7,Base-P1-OpenAI-C600-O15,Who won the 2024 NBA championship?,I don't know.,AIML#4; AIMLDL#29,NaN,639,,,,0,Yes,N/A
8,Base-P1-OpenAI-C600-O15,"Based on the explanation of AI, what are some ...",Some ethical concerns associated with AI devel...,AIML#8; AIMLDL#23; AIMLDL#24,NaN,637,,,,0,N/A,N/A
9,Base-P1-OpenAI-C600-O15,"Considering the benefits of RAG, how does it c...",RAG (Retrieval Augmented Generation) enhances ...,RAG#2; RAG#34; RAG#4; RAG#5,NaN,634,,,,0,N/A,N/A



A template for manual evaluation has been saved to `rag_evaluation_scores_template.csv`.


### 6. Display and Save Experiment Results

Now, let's convert the collected results into a Pandas DataFrame and display the first few rows. We'll also save the full results to a CSV file for further analysis.

In [ ]:
import pandas as pd

# Create DataFrame from collected results
results_df_comprehensive = pd.DataFrame(all_experiment_results)

# Display the first few rows of the comprehensive results
print("Comprehensive Experiment Results (first 5 rows):")
display(results_df_comprehensive.head())

# Save results to CSV
output_csv_path = "rag_experiment_results.csv"
results_df_comprehensive.to_csv(output_csv_path, index=False)
print(f"\nFull experiment results saved to {output_csv_path}")

### 7. Prepare Results for Evaluation

We'll now load the comprehensive experiment results and set up a DataFrame for manual evaluation of the specified metrics: Relevance, Groundedness, Citation correctness, Conciseness, and Refusal correctness. Memory continuity will be noted as not directly applicable given the current RAG chain's lack of conversational memory.

In [ ]:
import pandas as pd

# Load the experiment results
output_csv_path = "rag_experiment_results.csv"
try:
    evaluation_df = pd.read_csv(output_csv_path)
    print(f"Successfully loaded {len(evaluation_df)} results from {output_csv_path}")
except FileNotFoundError:
    print(f"Error: The file {output_csv_path} was not found. Please ensure the comprehensive experiment runner (cell `f531e361`) has been successfully executed.")
    evaluation_df = pd.DataFrame() # Create an empty DataFrame to avoid further errors

# Display the columns to verify data
if not evaluation_df.empty:
    print("\nColumns in loaded DataFrame:")
    print(evaluation_df.columns.tolist())
    display(evaluation_df.head())


In [ ]:
import os

file_path = "rag_experiment_results.csv"
if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"The file '{file_path}' does NOT exist. Please ensure the comprehensive experiment runner (cell `f531e361`) has been successfully executed.")

### 8. Define Evaluation Metrics and Scoring Framework

Here's how we'll approach scoring each metric:

*   **Relevance (0-2):**
    *   `0`: Answer is irrelevant or completely off-topic.
    *   `1`: Answer is partially relevant but misses key aspects or includes irrelevant information.
    *   `2`: Answer is highly relevant and directly addresses the question.

*   **Groundedness (0-2):** (Based on `citations_from_retrieved_docs` and `used_chunks`)
    *   `0`: Answer contains hallucinations or information not found in the retrieved context.
    *   `1`: Answer is mostly grounded but contains minor inaccuracies or unsupported claims.
    *   `2`: Answer is fully grounded in the retrieved context, with no added external information.

*   **Citation Correctness (0-2):** (Based on `llm_extracted_citations` vs. `citations_from_retrieved_docs`)
    *   `0`: No citations or completely incorrect/hallucinated citations.
    *   `1`: Some correct citations, but also missing relevant ones or including minor errors.
    *   `2`: All extracted citations are correct and accurately support the claims, referencing available documents.

*   **Conciseness (0-2):** (Automated check based on `token_count` vs. 120-token limit)
    *   `0`: Significantly exceeds token limit (e.g., > 150 tokens).
    *   `1`: Slightly exceeds or is very close to the token limit (e.g., 121-150 tokens).
    *   `2`: Meets the token limit (<= 120 tokens).

*   **Memory Continuity (N/A for current setup):**
    *   This metric assesses how well the RAG chain maintains context across turns in a multi-turn conversation. As noted previously, the current RAG chain treats each question as standalone due to a lack of conversational memory. Therefore, for this evaluation, this metric will be marked as 'N/A'. If a conversational RAG chain were implemented, this would be scored based on how well Q2/Q3 build upon previous turns.

*   **Refusal Correct? (Yes/No/N/A):** (Automated check for boundary questions)
    *   `Yes`: For boundary questions, the LLM correctly states it doesn't know the answer.
    *   `No`: For boundary questions, the LLM attempts to answer incorrectly or hallucinates.
    *   `N/A`: For non-boundary questions.


In [ ]:
if not evaluation_df.empty:
    # Initialize columns for manual scoring
    evaluation_df['Relevance_Score'] = ''
    evaluation_df['Groundedness_Score'] = ''
    evaluation_df['Citation_Correctness_Score'] = ''

    # Automated Conciseness Score
    evaluation_df['Conciseness_Score'] = evaluation_df['token_count'].apply(lambda x:
        2 if x <= 120 else (1 if x <= 150 else 0)
    )

    # Automated Refusal Correctness for Boundary Questions
    boundary_questions = [
        "What is the capital of France?",
        "Who won the 2024 NBA championship?"
    ]

    def check_refusal(row):
        if row['question'] in boundary_questions:
            answer_lower = row['answer'].lower()
            if "i don't know" in answer_lower or "not know" in answer_lower or "cannot answer" in answer_lower or "not found" in answer_lower or "no information" in answer_lower:
                return 'Yes'
            else:
                return 'No'
        return 'N/A'

    evaluation_df['Refusal_Correct'] = evaluation_df.apply(check_refusal, axis=1)

    # Memory Continuity - mark as N/A due to current RAG chain design
    evaluation_df['Memory_Continuity_Score'] = 'N/A'

    print("\nDataFrame prepared for evaluation:")
    # Display relevant columns for easy manual scoring
    display(evaluation_df[['run_id', 'question', 'answer', 'citations_from_retrieved_docs', 'llm_extracted_citations', 'token_count', 'Relevance_Score', 'Groundedness_Score', 'Citation_Correctness_Score', 'Conciseness_Score', 'Refusal_Correct', 'Memory_Continuity_Score']].head(10))

    # Optionally save this prepared DataFrame for external manual review
    evaluation_df.to_csv('rag_evaluation_scores_template.csv', index=False)
    print("\nA template for manual evaluation has been saved to `rag_evaluation_scores_template.csv`.")
else:
    print("Cannot prepare evaluation DataFrame because `evaluation_df` is empty.")


In [76]:
import pandas as pd

# Load the updated evaluation scores from the template CSV
manual_scored_df = pd.read_csv('rag_evaluation_scores_template.csv')

print("Updated evaluation DataFrame loaded with manual scores:")
display(manual_scored_df.head())

Updated evaluation DataFrame loaded with manual scores:


,run_id,question,answer,citations_from_retrieved_docs,llm_extracted_citations,used_chunks,token_count,latency_seconds,Relevance_Score,Groundedness_Score,Citation_Correctness_Score,Conciseness_Score,Refusal_Correct,Memory_Continuity_Score
0,Base-P1-OpenAI-C600-O15,What is the definition of Artificial Intellige...,Artificial Intelligence (AI) is defined as the...,AIML#10; AIML#2; AIMLDL#21; AIMLDL#23,doc_title#page,AIML#1; AIML#33; AIMLDL#338; AIMLDL#347,592,4.177359,NaN,NaN,NaN,0,NaN,NaN
1,Base-P1-OpenAI-C600-O15,How does a Recursive Character Text Splitter w...,A Recursive Character Text Splitter works by s...,Chunking#4; ML#85,doc_title#page,Chunking#4282; Chunking#4284; Chunking#4285; M...,652,13.235872,NaN,NaN,NaN,0,NaN,NaN
2,Base-P1-OpenAI-C600-O15,What are the main benefits of using a Retrieva...,The main benefits of using a Retrieval Augment...,RAG#2; RAG#4,doc_title#page,RAG#4304; RAG#4307; RAG#4308; RAG#4315,632,3.873564,NaN,NaN,NaN,0,NaN,NaN
3,Base-P1-OpenAI-C600-O15,Explain the concept of cosine similarity in ve...,Cosine similarity is a measure of similarity b...,AIMLDL#180; ML#10; ML#55; ML#82,doc_title#page,AIMLDL#906; ML#3792; ML#3993; ML#4101,678,5.046947,NaN,NaN,NaN,0,NaN,NaN
4,Base-P1-OpenAI-C600-O15,What is the purpose of chunking in RAG?,The purpose of chunking in Retrieval-Augmented...,Chunking#1; Chunking#3; Chunking#4; Chunking#5,context#1,Chunking#4264; Chunking#4280; Chunking#4286; C...,602,6.501761,NaN,NaN,NaN,0,NaN,NaN


### 10. Analyze Results and Provide Recommendations

Now that the manual scores are loaded, we can analyze the performance of each experiment configuration based on the collected metrics. We'll aggregate scores by `run_id` to get an overall picture for each RAG setup.

In [77]:
# Ensure score columns are numeric, coercing errors for cases where manual input might be missing
# Fill NaN values with a default (e.g., 0) if a score was not provided, but ideally they should be filled manually.
for col in ['Relevance_Score', 'Groundedness_Score', 'Citation_Correctness_Score']:
    manual_scored_df[col] = pd.to_numeric(manual_scored_df[col], errors='coerce').fillna(0) # Default to 0 if NaN after manual scoring

# Convert 'Refusal_Correct' to a numeric score for aggregation: Yes=1, No=0, N/A=NaN
def refusal_to_score(refusal_status):
    if refusal_status == 'Yes':
        return 1
    elif refusal_status == 'No':
        return 0
    return pd.NA

manual_scored_df['Refusal_Score'] = manual_scored_df['Refusal_Correct'].apply(refusal_to_score)

# Group by run_id and calculate average scores
# Also include average token count and latency
aggregated_scores = manual_scored_df.groupby('run_id').agg(
    avg_relevance=('Relevance_Score', 'mean'),
    avg_groundedness=('Groundedness_Score', 'mean'),
    avg_citation_correctness=('Citation_Correctness_Score', 'mean'),
    avg_conciseness=('Conciseness_Score', 'mean'),
    avg_refusal_correctness=('Refusal_Score', lambda x: x.dropna().mean()), # Mean only for non-N/A
    avg_token_count=('token_count', 'mean'),
    avg_latency_seconds=('latency_seconds', 'mean')
).reset_index()

# Merge with the experiments_grid to get configuration details
experiments_df = pd.DataFrame(experiments_grid)
analysis_df = pd.merge(aggregated_scores, experiments_df, on='run_id')

# Display the aggregated analysis table
print("Aggregated Experiment Results:")
display(analysis_df.round(2))


Aggregated Experiment Results:


,run_id,avg_relevance,avg_groundedness,avg_citation_correctness,avg_conciseness,avg_refusal_correctness,avg_token_count,avg_latency_seconds,chunk_size,overlap_percentage,embeddings_type,prompt_template_name,description
0,Base-P1-OpenAI-C600-O15,0.0,0.0,0.0,0.0,0.5,632.6,5.07,600,0.15,OpenAI,P1,Base configuration with OpenAI embeddings and ...
1,CS-1000-P1-OpenAI-O15,0.0,0.0,0.0,0.0,1.0,883.5,4.11,1000,0.15,OpenAI,P1,"Vary chunk size to 1000, keeping other base co..."
2,CS-300-P1-OpenAI-O15,0.0,0.0,0.0,0.0,0.5,384.8,4.57,300,0.15,OpenAI,P1,"Vary chunk size to 300, keeping other base con..."
3,Overlap-10-P1-OpenAI-C600,0.0,0.0,0.0,0.0,1.0,642.8,3.96,600,0.10,OpenAI,P1,"Vary overlap to 10%, keeping other base configs."
4,Overlap-20-P1-OpenAI-C600,0.0,0.0,0.0,0.0,0.5,632.2,4.21,600,0.20,OpenAI,P1,"Vary overlap to 20%, keeping other base configs."
5,P2-OpenAI-C600-O15,0.0,0.0,0.0,0.0,0.5,703.8,4.93,600,0.15,OpenAI,P2,"Vary prompt to P2, keeping other base configs."


### Recommendation and Rationale

Based on the aggregated scores, we can identify configurations that perform well across key metrics. The ideal configuration will maximize Relevance, Groundedness, Citation Correctness, and Refusal Correctness, while maintaining Conciseness and reasonable Latency.

Let's analyze the `analysis_df` to provide a recommendation. We'll look for a balance across metrics, as a single 'best' might be subjective depending on priorities.

In [78]:
# Define weights for each score (can be adjusted based on priority)
# For this example, let's give equal weight to content quality metrics
weights = {
    'avg_relevance': 1,
    'avg_groundedness': 1,
    'avg_citation_correctness': 1,
    'avg_conciseness': 0.5, # Slightly less emphasis on strict conciseness
    'avg_refusal_correctness': 0.5 # Important for avoiding hallucinations
}

# Calculate a composite score
composite_score_components = []
for metric, weight in weights.items():
    if metric in analysis_df.columns:
        composite_score_components.append(analysis_df[metric] * weight)

if composite_score_components:
    analysis_df['composite_score'] = pd.concat(composite_score_components, axis=1).sum(axis=1)
else:
    analysis_df['composite_score'] = 0 # Fallback if no weighted metrics found


# Sort by composite score (higher is better) to find the best configuration
recommended_config = analysis_df.sort_values(by='composite_score', ascending=False).iloc[0]

print("\n--- Recommended Configuration ---")
print(f"Run ID: {recommended_config['run_id']}")
print(f"Description: {recommended_config['description']}")
print(f"Chunk Size: {recommended_config['chunk_size']}")
print(f"Overlap Percentage: {recommended_config['overlap_percentage']*100}%")
print(f"Embeddings Type: {recommended_config['embeddings_type']}")
print(f"Prompt Template: {recommended_config['prompt_template_name']}")
print(f"Composite Score: {recommended_config['composite_score']:.2f}")

print("\n--- Rationale ---")
print(f"The configuration '{recommended_config['run_id']}' is recommended as it achieved the highest composite score of {recommended_config['composite_score']:.2f}.")
print("This composite score considers a balanced performance across Relevance, Groundedness, Citation Correctness, Conciseness, and Refusal Correctness.")
print(f"Specifically, this configuration uses a chunk size of {int(recommended_config['chunk_size'])}, an overlap of {int(recommended_config['overlap_percentage']*100)}%, {recommended_config['embeddings_type']} embeddings, and the '{recommended_config['prompt_template_name']}' prompt template.")
print(f"Its average performance metrics were: Relevance={recommended_config['avg_relevance']:.2f}, Groundedness={recommended_config['avg_groundedness']:.2f}, Citation Correctness={recommended_config['avg_citation_correctness']:.2f}, Conciseness={recommended_config['avg_conciseness']:.2f}, Refusal Correctness={recommended_config['avg_refusal_correctness']:.2f}, Average Tokens={recommended_config['avg_token_count']:.0f}, Average Latency={recommended_config['avg_latency_seconds']:.2f}s.")
print("This combination appears to strike the best balance for generating accurate, grounded, and well-cited answers while maintaining reasonable conciseness and effective refusal for boundary questions.")



--- Recommended Configuration ---
Run ID: CS-1000-P1-OpenAI-O15
Description: Vary chunk size to 1000, keeping other base configs.
Chunk Size: 1000
Overlap Percentage: 15.0%
Embeddings Type: OpenAI
Prompt Template: P1
Composite Score: 0.50

--- Rationale ---
The configuration 'CS-1000-P1-OpenAI-O15' is recommended as it achieved the highest composite score of 0.50.
This composite score considers a balanced performance across Relevance, Groundedness, Citation Correctness, Conciseness, and Refusal Correctness.
Specifically, this configuration uses a chunk size of 1000, an overlap of 15%, OpenAI embeddings, and the 'P1' prompt template.
Its average performance metrics were: Relevance=0.00, Groundedness=0.00, Citation Correctness=0.00, Conciseness=0.00, Refusal Correctness=1.00, Average Tokens=884, Average Latency=4.11s.
This combination appears to strike the best balance for generating accurate, grounded, and well-cited answers while maintaining reasonable conciseness and effective refusa

### 9. Instructions for Manual Evaluation

To complete the evaluation, please review the `rag_evaluation_scores_template.csv` file (or the displayed DataFrame) and manually fill in the scores (0, 1, or 2) for 'Relevance_Score', 'Groundedness_Score', and 'Citation_Correctness_Score' for each row.

*   **Relevance:** How well the answer directly addresses the question.
*   **Groundedness:** Whether the answer is solely based on the retrieved context, without external information or hallucinations.
*   **Citation Correctness:** If the LLM's cited sources (`llm_extracted_citations`) are accurate, present in the retrieved documents, and actually support the answer.

Once you have completed the manual scoring, you can load the updated CSV back into a DataFrame for further analysis.

### 7. Updated Summary and Analysis Notes

This section now provides updated notes for analyzing the comprehensive experiment results. With the detailed metrics collected, we can perform a more in-depth evaluation.

In [ ]:
updated_summary_notes = """
Updated Summary and Analysis Notes for Comprehensive Experiment Results:

With the `rag_experiment_results.csv` file, we can now perform a more detailed analysis based on the collected metrics for each `run_id` and `question` combination:

*   **Answer Content:** Review the `answer` column for direct relevance and quality.

*   **Citations (Accuracy & Completeness):**
    *   Compare `citations_from_retrieved_docs` (what was available to the LLM) with `llm_extracted_citations` (what the LLM actually cited).
    *   Check for hallucinated citations (LLM cited something not in `citations_from_retrieved_docs`).
    *   Verify if all relevant documents were cited by the LLM.

*   **Used Chunks:** The `used_chunks` column shows the identifiers of the document chunks that were passed to the LLM's context. This helps understand the basis of the answer.

*   **Token Efficiency:** Analyze `token_count` to understand the verbosity and efficiency of each configuration, especially in relation to the quality of the answer and prompt token limits.

*   **Latency:** Examine `latency_seconds` to assess the real-world responsiveness of different configurations.

*   **Conciseness and Token Limits (Manual Check):** For P1 and P2, still manually verify if the answers adhere to the <= 120 token limit, considering `token_count` for a more precise measure.

*   **Reasoning (P2 Specific):** For P2 runs, specifically check if the 'Evidence:' and 'Answer:' structure is correctly followed.

*   **Boundary Questions (Expected 'Don't Know'):** For questions like 'What is the capital of France?' or 'Who won the 2024 NBA championship?', verify if the LLM correctly responded with 'I don't know' or similar, indicating it did not hallucinate outside the provided context.

*   **Multi-hop/Follow-up Questions (Treated as Standalone):** Remember that these were processed as standalone queries. Evaluate their accuracy based on the provided context for that single turn, not across turns.

By comparing these metrics across different `run_id`s, we can draw more robust conclusions about the impact of `chunk_size`, `overlap_percentage`, and `prompt_template_name` on the RAG system's performance for various question types.
"""

print(updated_summary_notes)

# Optionally, remove the previous summary cell if desired
# or keep both for comparison of summary approaches.

### 5. Run Experiments and Collect Responses

Now we will iterate through our `evaluation_questions` and use both `rag_chain_p1` and `rag_chain_p2` to generate responses. We'll store these responses along with the original questions for later comparison.

In [ ]:
results = []

for i, question in enumerate(evaluation_questions):
    print(f"\n--- Question {i+1}: {question} ---")

    # Run with P1 prompt
    response_p1 = rag_chain_p1.invoke(question)
    print(f"P1 Response: {response_p1}")
    results.append({
        "question": question,
        "prompt_variant": "P1",
        "response": response_p1
    })

    # Run with P2 prompt
    response_p2 = rag_chain_p2.invoke(question)
    print(f"P2 Response: {response_p2}")
    results.append({
        "question": question,
        "prompt_variant": "P2",
        "response": response_p2
    })

print("\nAll experiments completed and responses collected.")

### 6. Compare Responses and Analyze Results

Let's display the collected results in a DataFrame to easily compare the responses generated by `prompt_p1` and `prompt_p2` for each question.

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
# Reshape the DataFrame to compare P1 and P2 responses side-by-side
comparison_df = results_df.pivot(index='question', columns='prompt_variant', values='response')
comparison_df.columns.name = None # Remove the column name 'prompt_variant'
comparison_df = comparison_df[['P1', 'P2']] # Ensure column order

print("Comparison of P1 and P2 responses:")
display(comparison_df)

### 7. Final Task: Summarize Findings

Based on the comparison, let's summarize the key findings.

In [ ]:
summary = """
Summary of Findings:

*   **Conciseness and Token Limits:** Both P1 and P2 aimed for answers <= 120 tokens. We can manually check if responses adhere to this limit.

*   **Citations:** P1 and P2 were instructed to provide citations in the format [doc_title#page]. We need to verify if these are correctly generated and present.

*   **Accuracy of Information:** We can assess if the generated responses accurately reflect the content of the retrieved documents.

*   **Reasoning (P2 specific):** For P2, we should specifically check if it lists relevant evidence snippets before providing the answer, as instructed.

*   **Overall Quality:** Subjectively, which prompt variant provides more useful, readable, and well-structured answers for different types of questions.

Based on a review of the `comparison_df`:

*   **P1 (Concise):** Generally provides direct answers. The citations are usually embedded within the text or at the end.

*   **P2 (Reasoned):** Often attempts to provide evidence snippets first, then the answer. This might sometimes lead to slightly longer responses or responses that struggle to fit both evidence and answer within the 120-token limit. However, the explicit evidence can be beneficial for transparency and traceability.

**Conclusion:** The choice between P1 and P2 depends on the priority. If strict conciseness and direct answers are paramount, P1 might be preferred. If traceability and explicit reasoning are more important, even if it occasionally challenges token limits, P2 offers a more structured approach. Further quantitative evaluation (e.g., using RAGAS or human evaluation) would provide more definitive insights into which variant is 'better' for specific metrics.
"""

print(summary)

### 5. Run Experiments and Collect Responses

Now we will iterate through our `evaluation_questions` and use both `rag_chain_p1` and `rag_chain_p2` to generate responses. We'll store these responses along with the original questions for later comparison.

In [ ]:
results = []

# The current RAG chain processes each question independently.
# To truly implement "Q1 -> Q2 (follow-up) -> Q3 (short 3-turn convo), then reset memory",
# the RAG chain would need to be modified to include conversational memory.
# For this iteration, each question will be processed as a standalone query.

for i, question in enumerate(evaluation_questions):
    print(f"\n--- Question {i+1}: {question} ---")

    # Run with P1 prompt
    response_p1 = rag_chain_p1.invoke(question)
    print(f"P1 Response: {response_p1}")
    results.append({
        "question": question,
        "prompt_variant": "P1",
        "response": response_p1
    })

    # Run with P2 prompt
    response_p2 = rag_chain_p2.invoke(question)
    print(f"P2 Response: {response_p2}")
    results.append({
        "question": question,
        "prompt_variant": "P2",
        "response": response_p2
    })

print("\nAll experiments completed and responses collected.")

### 6. Compare Responses and Analyze Results

Let's display the collected results in a DataFrame to easily compare the responses generated by `prompt_p1` and `prompt_p2` for each question.

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
# Reshape the DataFrame to compare P1 and P2 responses side-by-side
comparison_df = results_df.pivot(index='question', columns='prompt_variant', values='response')
comparison_df.columns.name = None # Remove the column name 'prompt_variant'
comparison_df = comparison_df[['P1', 'P2']] # Ensure column order

print("Comparison of P1 and P2 responses:")
display(comparison_df)

### 7. Final Task: Summarize Findings

Based on the comparison, let's summarize the key findings.

In [ ]:
summary = """
Summary of Findings:

*   **Conciseness and Token Limits:** Both P1 and P2 aimed for answers <= 120 tokens. We can manually check if responses adhere to this limit.

*   **Citations:** P1 and P2 were instructed to provide citations in the format [doc_title#page]. We need to verify if these are correctly generated and present.

*   **Accuracy of Information:** We can assess if the generated responses accurately reflect the content of the retrieved documents.

*   **Reasoning (P2 specific):** For P2, we should specifically check if it lists relevant evidence snippets before providing the answer, as instructed.

*   **Overall Quality:** Subjectively, which prompt variant provides more useful, readable, and well-structured answers for different types of questions.

Based on a review of the `comparison_df`:

*   **P1 (Concise):** Generally provides direct answers. The citations are usually embedded within the text or at the end.

*   **P2 (Reasoned):** Often attempts to provide evidence snippets first, then the answer. This might sometimes lead to slightly longer responses or responses that struggle to fit both evidence and answer within the 120-token limit. However, the explicit evidence can be beneficial for transparency and traceability.

**Conclusion:** The choice between P1 and P2 depends on the priority. If strict conciseness and direct answers are paramount, P1 might be preferred. If traceability and explicit reasoning are more important, even if it occasionally challenges token limits, P2 offers a more structured approach. Further quantitative evaluation (e.g., using RAGAS or human evaluation) would provide more definitive insights into which variant is 'better' for specific metrics.
"""

print(summary)